# Generate SQL Insert Statements
### Import necessary packages

In [13]:
import pandas as pd
import os

In [14]:
# Read the cleaned csv files
movies = pd.read_csv("../data/clean/movies_clean.csv")
boxoffice_df = pd.read_csv("../data/clean/boxoffice_clean.csv")
rotten_df = pd.read_csv("../data/clean/rotten_clean.csv")

### Create `create_tables.sql` file

In [15]:
create_sql = """
-- Drop tables if they exist 
BEGIN
   EXECUTE IMMEDIATE 'DROP TABLE rt_reviews PURGE';
EXCEPTION
   WHEN OTHERS THEN
      IF SQLCODE != -942 THEN RAISE; END IF;
END;
/

BEGIN
   EXECUTE IMMEDIATE 'DROP TABLE box_office PURGE';
EXCEPTION
   WHEN OTHERS THEN
      IF SQLCODE != -942 THEN RAISE; END IF;
END;
/

BEGIN
   EXECUTE IMMEDIATE 'DROP TABLE movie_genres PURGE';
EXCEPTION
   WHEN OTHERS THEN
      IF SQLCODE != -942 THEN RAISE; END IF;
END;
/

BEGIN
   EXECUTE IMMEDIATE 'DROP TABLE movies PURGE';
EXCEPTION
   WHEN OTHERS THEN
      IF SQLCODE != -942 THEN RAISE; END IF;
END;
/

PURGE RECYCLEBIN;

-- Create movies table
CREATE TABLE movies (
    tconst VARCHAR2(20) PRIMARY KEY,
    norm_title VARCHAR2(255),
    title_type VARCHAR2(50),
    start_year NUMBER(4),
    runtime_minutes NUMBER,
    imdb_rating NUMBER(3,1)
);

-- Movie genres table (1-to-many relationship)
CREATE TABLE movie_genres (
    tconst VARCHAR2(20) NOT NULL,
    genre VARCHAR2(100),
    PRIMARY KEY (tconst, genre),
    FOREIGN KEY (tconst) REFERENCES movies(tconst) ON DELETE CASCADE
);

-- Create box_office table
CREATE TABLE box_office (
    box_id NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    norm_title VARCHAR2(255),
    genre VARCHAR2(100),
    year NUMBER(4),
    worldwide_revenue NUMBER
);

-- Create rt_reviews table
CREATE TABLE rt_reviews (
    review_id NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    norm_title VARCHAR2(255),
    tomatometer_rating NUMBER,
    critics_consensus VARCHAR2(4000)
);
"""

with open("../sql/create_tables.sql", "w") as f:
    f.write(create_sql)

### Create helper functions to handel NULL values when building INSERT statements

In [16]:
def safe_int(value):
    """
    Convert numeric string to int or return NULL if missing.
    """
    if pd.isna(value) or value == "\\N":
        return "NULL"
    return int(value)

def safe_float(value):
    """
    Convert numeric string to float or return NULL if missing.
    """
    if pd.isna(value) or value == "\\N":
        return "NULL"
    return float(value)

def safe_str(value):
    """
    Returns a safe string for SQL parsing, or NULL if value is missing.
    Single quotes in a string are doubled to avoid confusion of where string ends.
    ex. "Schindler's List" becomes "'Schindler''s List'"
    """
    if pd.isna(value):
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"


### Insert values into `movies` table

In [17]:
sql_lines = []

for _, row in movies.iterrows():
    tconst      = safe_str(row["tconst"])
    norm_title  = safe_str(row["norm_title"])
    title_type  = safe_str(row["titleType"])
    start_year  = safe_int(row["startYear"])
    runtime     = safe_int(row["runtimeMinutes"])
    imdb_rating = safe_float(row["averageRating"])

    sql = f"""INSERT INTO movies (tconst, norm_title, title_type, start_year, runtime_minutes, imdb_rating) 
                VALUES ({tconst}, {norm_title}, {title_type}, {start_year}, {runtime}, {imdb_rating});"""
    sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/movies_inserts.sql", "w", encoding = "utf-8") as f:
    f.write("\n".join(sql_lines))

print("movies_inserts.sql generated")

movies_inserts.sql generated


### Insert values into `movie_genres` table

In [18]:
sql_lines = []

for _, row in movies.iterrows():
    if pd.isna(row["genres"]): # If there are NA values in genre column, continue parsing
        continue
    for genre in row["genres"].split(","):
        genre_clean = safe_str(genre.strip())
        tconst = safe_str(row["tconst"])
        sql = f"""INSERT INTO movie_genres (tconst, genre) 
                    VALUES ({tconst}, {genre_clean});"""
        sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/movies_genres_inserts.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql_lines))

print("movies_genres_inserts.sql generated")

movies_genres_inserts.sql generated


### Insert values into `box_office` table

In [19]:
sql_lines = ["SET DEFINE OFF;"] # This prevents the "&" character from being executed in Oracle

for _, row in boxoffice_df.iterrows():
    norm_title = safe_str(row["norm_title"])
    genre      = safe_str(row["Genres"])
    year       = safe_int(row["Year"])
    worldwide  = safe_float(row["$Worldwide"])

    sql = f"""INSERT INTO box_office (norm_title, genre, year, worldwide_revenue) 
                VALUES ({norm_title}, {genre}, {year}, {worldwide});"""
    sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/boxoffice_inserts.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql_lines))

print("boxoffice_inserts.sql generated")

boxoffice_inserts.sql generated


### Insert values into `rt_reviews` table

In [20]:
sql_lines = ["SET DEFINE OFF;"]  # This prevents the "&" character from being executed in Oracle

for _, row in rotten_df.iterrows():
    norm_title  = safe_str(row["norm_title"])
    tomatometer = safe_float(row["tomatometer_rating"])
    consensus   = safe_str(row["critics_consensus"])

    sql = f"""INSERT INTO rt_reviews (norm_title, tomatometer_rating, critics_consensus) 
                VALUES ({norm_title}, {tomatometer}, {consensus});"""
    sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/rt_reviews_inserts.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql_lines))

print("rt_reviews_inserts.sql generated")

rt_reviews_inserts.sql generated
